# 12주 · 평균 가설검정·분석 조건 선택

공학통계학 CE202 · 2027학년도 1학기

슬라이드의 개념·숫자·풀이 순서와 연결한 **새 병행 실습**. 원본 Colab 복원본이 아닌 신규 자료

**목표**: 통계 원리와 도구의 기능 이해 → 좋은 AI 질문·지시 → 계산·가정·해석의 검토

**수업**: 2027-05-18 가설검정 (1) · 온라인 영상강의 / 2027-05-20 가설검정 (2) · 대면

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE202-statistics-for-engineers/week12/CE202_W12.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE202-statistics-for-engineers/week12)

슬라이드 확인 → 실행 전 예측 → 작은 손계산 → 코드 실행 → 조건 변경 → AI 요청 수정 → 결과 검토. 패키지 문법 암기보다 입력·정의·가정·출력의 의미 파악

셀 실행: Shift+Enter. 아래 코드의 None은 직접 채울 항목. 원본과 변경 사본의 결과를 구분

## 0. 실행 환경

필요한 패키지의 최소 버전 확인. 기본 코드와 보조 함수는 이번 주 실습에 필요한 도구. 세부 구현 암기 불필요

In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.16", "matplotlib": "3.10", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math, tempfile, base64, zipfile, io
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
WORK = Path(tempfile.mkdtemp(prefix="ce202-"))
rng = np.random.default_rng(2027)

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 수치 자료와 최소 표본수 확인 필요")
    return x

def probabilities(values):
    p = vector(values)
    if np.any(p < 0) or not np.isclose(p.sum(), 1):
        raise ValueError("확률의 비음수성·합계 1 확인 필요")
    return p

def mean_ci(values, confidence=0.95):
    x = vector(values, 2)
    if not 0 < confidence < 1:
        raise ValueError("신뢰수준은 0과 1 사이")
    se = x.std(ddof=1) / np.sqrt(len(x))
    q = stats.t.ppf((1 + confidence) / 2, df=len(x)-1)
    return {"n": len(x), "mean": x.mean(), "se": se,
            "df": len(x)-1, "low": x.mean()-q*se, "high": x.mean()+q*se}

def wilson_interval(successes, n, confidence=0.95):
    if not isinstance(n, (int, np.integer)) or n <= 0 or not 0 <= successes <= n:
        raise ValueError("0 ≤ 성공수 ≤ 양의 정수 표본수 확인 필요")
    if int(successes) != successes or not 0 < confidence < 1:
        raise ValueError("정수 성공수·신뢰수준 확인 필요")
    z = stats.norm.ppf((1+confidence)/2)
    phat = successes / n
    den = 1 + z*z/n
    center = (phat + z*z/(2*n)) / den
    half = z / den * np.sqrt(phat*(1-phat)/n + z*z/(4*n*n))
    return center-half, center+half

def anova_summary(groups):
    arrays = [vector(g, 2) for g in groups]
    if len(arrays) < 2:
        raise ValueError("두 집단 이상 필요")
    n = np.array([len(g) for g in arrays]); means = np.array([g.mean() for g in arrays])
    grand = np.concatenate(arrays).mean()
    ss_between = np.sum(n*(means-grand)**2)
    ss_within = sum(np.sum((g-g.mean())**2) for g in arrays)
    df_between = len(arrays)-1; df_within = n.sum()-len(arrays)
    if ss_within <= 0:
        raise ValueError("집단 내 변동이 0이면 일반적인 F 계산 불가")
    F = (ss_between/df_between) / (ss_within/df_within)
    return {"ss_between": ss_between, "ss_within": ss_within,
            "df_between": df_between, "df_within": df_within,
            "F": F, "p": stats.f.sf(F, df_between, df_within)}

def regression(x, y):
    x = vector(x, 3); y = vector(y, 3)
    if len(x) != len(y) or np.ptp(x) == 0:
        raise ValueError("같은 길이·서로 다른 x 값 확인 필요")
    fit = stats.linregress(x, y)
    fitted = fit.intercept + fit.slope*x
    residual = y-fitted
    sse = np.sum(residual**2); sst = np.sum((y-y.mean())**2)
    if sst <= 0:
        raise ValueError("y 변동이 0이면 일반적인 R² 해석 불가")
    return {"slope": fit.slope, "intercept": fit.intercept, "r2": 1-sse/sst,
            "sse": sse, "mse_residual": sse/(len(x)-2), "mse_prediction": sse/len(x),
            "p": fit.pvalue, "stderr": fit.stderr, "fitted": fitted, "residual": residual}


## S01. 원본 볼트의 z와 t 검정

**슬라이드 연결**: 2026_S1_Statistics_Wk12a_test1.pptx p15, 16, 21, 25, 36

**자료 구분**: 원본 예제·수치 기반

**먼저 예측**: 모표준편차와 표본표준편차를 구분하지 않으면 어떤 분포를 고르게 되는가?

계산 전에 분모·단위·조건을 한 줄로 기록

In [ ]:
z_stat=(10.5-10)/(1.2/np.sqrt(36)); z_p=2*stats.norm.sf(abs(z_stat))
t_stat=(10.4-10)/(.8/np.sqrt(16)); t_p=2*stats.t.sf(abs(t_stat),df=15)
print({"모표준편차 기지 z":z_stat,"p":z_p,"표본표준편차 t":t_stat,"t_p":t_p})
assert np.isclose(z_stat,2.5) and np.isclose(t_stat,2)
assert z_p<.05 and t_p>.05

**조건 변경**: 단측으로 바꾸려면 자료를 보기 전에 어떤 가설이 필요한지 기록

위 코드를 복사한 셀에서 한 조건씩 변경. 원본 셀은 보존

**AI 요청에 넣을 정보**: H0·H1·α·검정 방향·n·표준편차 출처를 명시

**결과 검토**: p값은 H0가 참일 확률이나 관측값 한 점의 확률이 아님

실행 결과·손계산 근거·수정 지시·채택 또는 거절 이유 기록

In [ ]:
# 변경한 코드·확인할 식을 이 셀에 추가


## S02. 원본 독립집단과 대응 자료

**슬라이드 연결**: 2026_S1_Statistics_Wk12b_test2.pptx p7, 14, 15, 16, 18, 23, 24, 25

**자료 구분**: 원본 예제·수치 기반

**먼저 예측**: 독립집단과 같은 사람의 전후를 같은 검정으로 처리해도 되는가?

계산 전에 분모·단위·조건을 한 줄로 기록

In [ ]:
n1=n2=9; mean1,mean2=186.,174.; sd1,sd2=11.,10.
sp2=((n1-1)*sd1**2+(n2-1)*sd2**2)/(n1+n2-2)
t_ind=(mean1-mean2)/np.sqrt(sp2*(1/n1+1/n2)); p_ind=2*stats.t.sf(abs(t_ind),16)
print("원본 스마트폰 사용시간(분), 등분산 독립t:",t_ind,p_ind)
before=np.array([75, 70, 88, 85, 95, 65, 87, 70, 95, 85]);after=np.array([80, 80, 90, 90, 90, 60, 90, 85, 90, 93])
paired_test=stats.ttest_rel(after,before,alternative="greater")
print("원본 교육 전후, 후−전 증가 단측:",paired_test)
assert np.isclose((after-before).mean(),3.3)
assert paired_test.pvalue>.05

**조건 변경**: 대응 자료를 독립검정으로 잘못 분석한 사본과 비교

위 코드를 복사한 셀에서 한 조건씩 변경. 원본 셀은 보존

**AI 요청에 넣을 정보**: 등분산 가정 유무·대응 ID·차이 부호·단측 가설을 명시한 분석 요청

**결과 검토**: 기각 못함과 H0 참의 입증을 구분. 등분산 근거가 없으면 Welch 등 검토

실행 결과·손계산 근거·수정 지시·채택 또는 거절 이유 기록

In [ ]:
# 변경한 코드·확인할 식을 이 셀에 추가


## 주차 확인

귀무가설·분포·p값의 의미 기록

입력 항목:

- `bolt_t_statistic`: 모표준편차 미지인 원본 볼트 예제의 t 검정통계량 · 수치
- `bolt_t_df`: 원본 볼트 t검정의 자유도 · 수치
- `non_rejection_proves_H0`: 기각 못함이 H0가 참이라는 입증인가? · Python `True`/`False`
- `p_value_is_probability_H0_true`: p값이 H0가 참일 확률인가? · Python `True`/`False`

공개 확인용 항목. 결과값만 복사한 경우 학습 완료로 판단 불가. 분석 가정·자료 출처·추론 범위는 아래 기록과 함께 검토

In [ ]:
answers = {'bolt_t_statistic': None, 'bolt_t_df': None, 'non_rejection_proves_H0': None, 'p_value_is_probability_H0_true': None}


In [ ]:
def check_answers(values):
    expected = {'bolt_t_statistic': 2.0, 'bolt_t_df': 15, 'non_rejection_proves_H0': False, 'p_value_is_probability_H0_true': False}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## AI 요청·검토 기록

다음 항목에 직접 기록. 글자 수 자동 채점 없음

1. 분석 질문·관측단위·자료 출처·변수·단위
2. 최초 AI 요청: 목적·조건·방법·출력 형식·금지할 가정
3. 실행 전 예상·손계산 근거
4. AI 출력 중 의심한 부분·재현 결과·수정 요청
5. 최종 결과와 원본의 일치 여부·분석 가정·불확실성·한계
6. AI 없이 설명 가능한 핵심 원리 한 가지

채택·수정·거절 중 하나를 선택하고 근거 제시. p값·R² 하나만으로 결론 확정 금지

In [ ]:
review_log = {
    "분석 질문·자료·단위": None,
    "최초 AI 요청": None,
    "예상·손계산": None,
    "의심한 부분·수정 요청": None,
    "최종 판단·가정·한계": None,
    "내 설명": None,
}
